# TinyLlama GPTQ (Kaggle, updated)

This notebook runs GPTQ-style layer quantization for TinyLlama and lets you choose where heavy math runs (`MATH_DEVICE`).

In [ ]:
!pip -q install -U transformers datasets accelerate safetensors tqdm

In [1]:
print("All required libraries have been installed successfully.")

All required libraries have been installed successfully.


In [ ]:
import os
import time
import numpy as np
import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# -------------------------
# Config
# -------------------------
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'        # model forward device
MATH_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'   # Hessian/quant math device

MODEL_PATH = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
OUTPUT_PATH = '/kaggle/working/tinyllama_gptq_nf4'
QUANT_LIST_ID = 4  # 1,2,3,4

LAYER_TYPES = (
    'self_attn.q_proj',
    'self_attn.k_proj',
    'self_attn.v_proj',
    'self_attn.o_proj',
    'mlp.gate_proj',
    'mlp.up_proj',
    'mlp.down_proj',
)

FAST_MODE = True
MAX_BLOCKS = 1 if FAST_MODE else None

DATASET = 'wikitext'
DATASET_CONFIG = 'wikitext-2-raw-v1'
CALIB_SPLIT = 'train'
CALIB_MAX_ROWS = 500 if FAST_MODE else 4000
BLOCK_SIZE = 128 if FAST_MODE else 256
BATCH_SIZE = 4 if FAST_MODE else 16
SEED = 555

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_float32_matmul_precision('high')

print('DEVICE:', DEVICE)
print('MATH_DEVICE:', MATH_DEVICE)
print('FAST_MODE:', FAST_MODE)

In [ ]:
def get_calibration_batch(dataset_name, dataset_config, split, tokenizer, block_size, batch_size, seed, max_rows=4000):
    torch.manual_seed(seed)
    np.random.seed(seed)

    ds = load_dataset(dataset_name, dataset_config, split=split)
    ds = ds.select(range(min(max_rows, len(ds))))
    texts = [t for t in ds['text'] if isinstance(t, str) and t.strip()]
    if not texts:
        raise ValueError('Calibration dataset returned no usable text rows.')

    corpus = '\n\n'.join(texts)
    ids = tokenizer(corpus, add_special_tokens=False, verbose=False)['input_ids']
    if len(ids) <= block_size:
        raise ValueError(f'Tokenized corpus too short ({len(ids)}) for block_size={block_size}.')

    all_ids = np.array(ids, dtype=np.int64)
    ix = torch.randint(len(all_ids) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(all_ids[i:i+block_size].copy()) for i in ix]).long()
    return x


def get_activations(model, x_tokens, target_layer):
    activations = []

    def hook(module, input, output):
        # keep GPU memory low by moving captured layer input to CPU
        activations.append(input[0].detach().cpu())

    handle = target_layer.register_forward_hook(hook)
    with torch.no_grad():
        model(x_tokens)
    handle.remove()

    return torch.cat(activations, dim=0)


def get_layer_weights(target_layer):
    W = target_layer.weight.detach().clone()
    b = target_layer.bias.detach().clone() if target_layer.bias is not None else None
    return W, b


def get_target_layer(model, block_index, layer_name):
    block = model.model.layers[block_index]
    if layer_name == 'self_attn.q_proj':
        return block.self_attn.q_proj
    if layer_name == 'self_attn.k_proj':
        return block.self_attn.k_proj
    if layer_name == 'self_attn.v_proj':
        return block.self_attn.v_proj
    if layer_name == 'self_attn.o_proj':
        return block.self_attn.o_proj
    if layer_name == 'mlp.gate_proj':
        return block.mlp.gate_proj
    if layer_name == 'mlp.up_proj':
        return block.mlp.up_proj
    if layer_name == 'mlp.down_proj':
        return block.mlp.down_proj
    raise ValueError(f'Unknown layer name: {layer_name}')


def quantize_with_hessian_per_row(W, H_inv, scale_multiplier=1.0, which_list=4):
    W_quant = W.clone().float()
    _, n_in = W.shape

    if which_list == 1:
        quant_list = W_quant.new_tensor([-16.0, -8.0, -4.0, -2.0, -1.0, -0.5, -0.25, 0.0, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0])
    elif which_list == 2:
        quant_list = W_quant.new_tensor([-6.0, -4.0, -3.0, -2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0])
    elif which_list == 3:
        quant_list = W_quant.new_tensor([-3.5, -3.0, -2.5, -2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5])
    elif which_list == 4:
        quant_list = W_quant.new_tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])
    else:
        raise ValueError('Select quant_list id in {1,2,3,4}.')

    quant_list_max = quant_list.abs().max()

    for i in tqdm(range(n_in), desc='Quant cols', leave=False):
        w_col = W_quant[:, i].clone()
        row_absmax = W_quant.abs().amax(dim=1).clamp_min(1e-8)
        s_vec = (row_absmax / quant_list_max) * scale_multiplier

        u = w_col / s_vec
        idx = torch.argmin((u.unsqueeze(1) - quant_list.unsqueeze(0)).abs(), dim=1)
        q_levels = quant_list[idx]
        w_q = q_levels * s_vec

        error = w_col - w_q
        if i < n_in - 1:
            update = error.unsqueeze(1) @ (H_inv[i, i+1:] / H_inv[i, i]).unsqueeze(0)
            W_quant[:, i+1:] -= update

        W_quant[:, i] = w_q

    return W_quant


def find_optimal_scale(W, H_inv, X_flat, Y_ref, which_list=4, fast_mode=True):
    best_mse = float('inf')
    best_scale = 1.0
    best_W_q = None

    test_scales = np.linspace(0.8, 1.2, 5) if fast_mode else np.linspace(0.7, 1.3, 25)
    for s_mult in tqdm(test_scales, desc='Scale search', leave=False):
        W_q_temp = quantize_with_hessian_per_row(W, H_inv, scale_multiplier=s_mult, which_list=which_list)
        with torch.no_grad():
            Y_temp = X_flat @ W_q_temp.t()
            mse = torch.nn.functional.mse_loss(Y_temp, Y_ref).item()

        if mse < best_mse:
            best_mse = mse
            best_scale = s_mult
            best_W_q = W_q_temp

    return best_W_q, best_mse, best_scale

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model_dtype = torch.float16 if DEVICE == 'cuda' else torch.float32

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=model_dtype, device_map=None)
model.to(DEVICE)
model.eval()

num_blocks = len(model.model.layers)
if MAX_BLOCKS is not None:
    num_blocks = min(num_blocks, MAX_BLOCKS)

target_layers = [(bi, ln) for bi in range(num_blocks) for ln in LAYER_TYPES]

x_tokens = get_calibration_batch(
    dataset_name=DATASET,
    dataset_config=DATASET_CONFIG,
    split=CALIB_SPLIT,
    tokenizer=tokenizer,
    block_size=BLOCK_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    max_rows=CALIB_MAX_ROWS,
).to(DEVICE)

print('Batch shape:', tuple(x_tokens.shape))
print('Calibration split:', CALIB_SPLIT)
print('Target layer count:', len(target_layers))

In [ ]:
for block_index, layer_name in tqdm(target_layers, desc='Target layers'):
    print(f'\n=== Quantizing block {block_index} / {layer_name} ===')
    target_layer = get_target_layer(model, block_index, layer_name)

    t0 = time.time()
    X_big = get_activations(model, x_tokens, target_layer=target_layer)
    print(f'get_activations sec: {time.time()-t0:.2f}')

    W_orig, _ = get_layer_weights(target_layer)
    x_dim = W_orig.shape[1]

    # Heavy GPTQ math device (cuda/cpu)
    X_flat = X_big.view(-1, x_dim).float().to(MATH_DEVICE)
    W_orig_math = W_orig.float().to(MATH_DEVICE)

    assert X_flat.ndim == 2
    assert W_orig_math.ndim == 2
    assert X_flat.shape[1] == W_orig_math.shape[1]

    with torch.no_grad():
        t1 = time.time()
        Y_ref = X_flat @ W_orig_math.t()
        H = X_flat.t() @ X_flat
        eps = 0.01 * torch.mean(torch.diag(H))
        H = H + eps * torch.eye(H.shape[0], device=H.device, dtype=H.dtype)
        H_inv = torch.inverse(H.float())
        print(f'Hessian+inverse sec: {time.time()-t1:.2f}')

        t2 = time.time()
        best_W_q, best_mse, best_scale = find_optimal_scale(
            W_orig_math,
            H_inv,
            X_flat,
            Y_ref,
            which_list=QUANT_LIST_ID,
            fast_mode=FAST_MODE,
        )
        print(f'scale_search sec: {time.time()-t2:.2f}')

    print(f'Best scale: {best_scale:.3f} | best MSE: {best_mse:.6f}')
    target_layer.weight.data.copy_(best_W_q.to(device=target_layer.weight.device, dtype=target_layer.weight.dtype))

    del X_big, X_flat, W_orig, W_orig_math, Y_ref, H, H_inv, best_W_q
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

In [ ]:
os.makedirs(OUTPUT_PATH, exist_ok=True)
model.save_pretrained(OUTPUT_PATH, safe_serialization=True)
tokenizer.save_pretrained(OUTPUT_PATH)
print('Saved quantized model + tokenizer to:', OUTPUT_PATH)